#"BIG DATA" 
Analisis de predictivo y descriptivo del dataset de MOVIES IMDB

In [0]:
%sql
SELECT * FROM movies_visualizacion_3

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#MINERIA DESCRIPTIVA

In [0]:
import pandas as pd
import seaborn as sns

data = spark.sql("select * from movies_visualizacion_3").toPandas()
data.head()

In [0]:
data.info()

In [0]:
#Corregir tipos de datos de object a category
for col in data.columns:
  if data[col].dtype == 'object':
    data[col] = data[col].astype('category')
data.info()

In [0]:
# Copia de los datos
dataCopia = data.copy()
dataCopia.head()

In [0]:
# Normalizacion de variables numéricas
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
variables_a_normalizar=['score', 'votes',  'gross',  'runtime'] # Variables Numericas
min_max_scaler.fit(dataCopia[variables_a_normalizar]) # Ajuste de parámetro
dataCopia[variables_a_normalizar]= min_max_scaler.transform(dataCopia[variables_a_normalizar])
dataCopia.head()

In [0]:
# Se crean dummies para las variables categóricas
dataCopia = pd.get_dummies(dataCopia, columns=['rating', 'genre', 'director', 'star','company'], drop_first=False, dtype=int) # drop_first elimina una dummie (en falso pq todas las caegoricas tienen mas de 2 variables)

#data = pd.get_dummies(data, columns=['Tipo'], drop_first=False,  dtype=int) #No borra dummie
dataCopia.head()

#Aprendizaje de Modelo
Metodo del codo/rodilla (usaremos codo ya que Rodilla es muy pesado computacionalmente)


In [0]:
# Método del codo para encontrar la mejor cantidad de clusters: inertia
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

ks = range(2, 20) # crear valores del 2 al 20
inertias = []

for k in ks:
    # Crear  modelo
    model = KMeans(n_clusters=k,max_iter=300)
    model.fit(dataCopia)
    inertias.append(model.inertia_)

# Graficar cantidad de clusters vs inertias
plt.plot(ks, inertias, '-o')
plt.xlabel('Numero de clusters, k')
plt.ylabel('inertia')
plt.xticks(ks)
plt.show()

#KMEANS

In [0]:
#Método de la rodilla: silueta
from sklearn import metrics

ks = range(2, 20) # crear valores del 2 al 20
siluetas = []

for k in ks:
    # Crear  modelo
    model = KMeans(n_clusters=k,max_iter=300)
    model.fit(dataCopia)
    sil=metrics.silhouette_score(dataCopia, model.labels_)
    siluetas.append(sil)

# Graficar cantidad de clusters vs inertias
plt.plot(ks, siluetas, '-o')
plt.xlabel('Numero de clusters, k')
plt.ylabel('silueta')
plt.xticks(ks)
plt.show()

In [0]:
# Creación de modelo de clustering con Kmeans
from sklearn.cluster import KMeans
k=18 #se estabiliza la silueta y luego el cambio no es tan grande para evitar exceso computacional
model = KMeans(n_clusters=k, max_iter=300)
model.fit(dataCopia)

#Evaluación del Modelo


*   Inertia: valor pequeño esperado
*   Silueta: valor positivo esperado, idealmente mayor a 0.5

In [0]:
# Evaluación
from sklearn import metrics

# Inertia: se require valor pequeño
print('Inercia o cohesión:', model.inertia_)

# Silueta: se requiere que sea positivo, ideal 0.5-1.0
sil=metrics.silhouette_score(dataCopia, model.labels_)
print('Silueta:',sil)

# Perfilamiento
### Descripción de Centroides

In [0]:
# Centroides de los cluster se convierten en un dataframe de pandas (no se están calculando, solo se convierten en un dataframe)
centroides=pd.DataFrame(model.cluster_centers_, columns=dataCopia.columns.values)
centroides.round(1)

In [0]:
# Se realiza una des-normalización centroides
centroides[variables_a_normalizar]=min_max_scaler.inverse_transform(centroides[variables_a_normalizar])
centroides.round(0)

#Perfilamiento:
Cluster 0

Perfil: peli PG-13, bien valorada (7.0), popularidad media-alta (~77k votos), taquilla media (~57M), dura ~112 min.

Lectura: blockbuster mediano de estudio, para público amplio, más serio que infantil.

Cluster 1

Perfil: peli R, score 6.0, votos ~87k, taquilla ~48M, 106 min.

Lectura: cine comercial para adultos (acción/crimen/drama), rendimiento correcto pero no masivo.

Cluster 2

Perfil: peli PG, score 6.0, votos ~82k, taquilla alta (~118M), 101 min.

Lectura: familiar/generalista que sí vendió; duración típica de familiar.

Cluster 3

Perfil: peli PG, score 7.0, votos ~100k, taquilla muy alta (~182M), 99 min.

Lectura: familiar / para todo público exitosa y bien valorada.

Cluster 4

Perfil: peli R, score 6.0, votos ~59k, taquilla baja-media (~33M), 102 min.

Lectura: título adulto mediano, menos visible.

Cluster 5

Perfil: peli PG-13, score 6.0, votos ~126k (bastante), taquilla ~98M, 103 min.

Lectura: acción/aventura/comercial de estudio, rendimiento sólido.

Cluster 6

Perfil: peli R, score 7.0, votos ~78k, taquilla ~30M, 113 min.

Lectura: película adulta mejor valorada que vendida.

Cluster 7

Perfil: peli R, score 7.0, votos ~96k, taquilla ~37M, 112 min.

Lectura: similar al 6 pero un poco más visible; drama/acción adulto.

Cluster 8

Perfil: peli PG-13, score 6.0, votos ~179k (muy popular), taquilla muy alta (~193M), 112 min.

Lectura: gran título comercial PG-13 que pegó fuerte.

Cluster 9

Perfil: peli PG, score 6.0, votos ~88k, taquilla ~106M, 105 min.

Lectura: familiar/comercial que funcionó bien.

Cluster 10

Perfil: peli R, score 7.0, votos ~90k, taquilla ~43M, 120 min (más larga).

Lectura: peli adulta más larga, buena valoración, venta moderada.

Cluster 11

Perfil: peli R, score 6.0, votos ~74k, taquilla ~37M, 101 min.

Lectura: película adulta estándar, rendimiento medio.

Cluster 12

Perfil: peli R, score 6.0, votos ~172k (muy vista), taquilla alta (~120M), 112 min.

Lectura: peli adulta grande: mucha gente la vio y recaudó bien.

Cluster 13

Perfil: peli PG-13, score 6.0, votos ~168k, taquilla muy alta (~211M), 111 min.

Lectura: super-taquillera PG-13.

Cluster 14

Perfil: peli PG-13, score 6.0, votos ~66k, taquilla ~65M, 103 min.

Lectura: comercial media, PG-13, de alcance moderado.

Cluster 15

Perfil: peli PG, score 6.0, votos ~39k (menos popular), taquilla ~57M, 102 min.

Lectura: familiar/para todo público más chica en visibilidad.

Cluster 16

Perfil: peli PG-13, score 6.0, votos ~266k (el más alto), taquilla muy muy alta (~326M), 117 min.

Lectura: mega-blockbuster PG-13.

In [0]:
# En el dataframe original, se adiciona el cluster asignado a cada registro
data['cluster']=model.labels_
data.head()

In [0]:
#Cantidad de datos en cada cluster
data["cluster"].value_counts().plot(kind='pie',autopct='%.0f%%')

In [0]:
#Para almacenar
# Almacenar resultados
data.to_excel('./resultadosMovies_Kmeans.xlsx')
centroides.to_excel('./centroidesMovies.xlsx')

#MINERIA PREDICTIVA

In [0]:
import pandas as pd
import seaborn as sns

data = spark.sql("select * from movies_visualizacion_3").toPandas()
data.head()

In [0]:
data.info()

In [0]:
#Corregir tipos de datos de object a category
for col in data.columns:
  if data[col].dtype == 'object':
    data[col] = data[col].astype('category')
data.info()

In [0]:
# Copia de los datos
dfcopy = data.copy()
dfcopy.head()

In [0]:
# Normalizacion de variables numéricas
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
variables_a_normalizar=[ 'votes',  'gross',  'runtime'] # Variables Numericas NO LA OBJETIVO PQ ES REGRESION
min_max_scaler.fit(dfcopy[variables_a_normalizar]) # Ajuste de parámetro
dfcopy[variables_a_normalizar]= min_max_scaler.transform(dfcopy[variables_a_normalizar])
dfcopy.head()

In [0]:
# Se crean dummies para las variables categóricas
dfcopy = pd.get_dummies(dfcopy, columns=['rating', 'genre', 'director', 'star','company'], drop_first=False, dtype=int) # drop_first elimina una dummie (en falso pq todas las caegoricas tienen mas de 2 variables)

dfcopy.head()

No hay labelencoder-> Variable objetivo ya es numérica

**Modelos Predictivos de Regresion**
Objetivo: Predecir la variable score (calificación) de cada película a partir de atributos numéricos y categóricos. Las variables categóricas se transformaron a dummies y las numéricas principales (votes, gross, runtime, year) se normalizaron, para que los modelos sensibles a escala (KNN y SVR) trabajen correctamente. El conjunto se dividió 70% entrenamiento / 30% prueba.

#División 70-30

In [0]:
#División 70-30
# División 70-30 score como variable objetivo
from sklearn.model_selection import train_test_split

# se quita el índice importado y la columna objetivo de X
cols_a_quitar = [c for c in ["Unnamed: 0", "score"] if c in dfcopy.columns]
X = dfcopy.drop(columns=cols_a_quitar)   # todas las dummies y numéricas útiles
Y = dfcopy["score"]                       # objetivo (regresión)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.30,     # 70/30
    random_state=42,    # reproducible
    shuffle=True        # baraja las filas
    # stratify=None     # para regresión no se usa estratificación
)

print("Shapes ->",
      "X_train:", X_train.shape, "| X_test:", X_test.shape,
      "| Y_train:", Y_train.shape, "| Y_test:", Y_test.shape)

# Boxplot rápido del objetivo en el conjunto de entrenamiento
Y_train.plot(kind='box', title='Distribución de score (train)')

In [0]:
#Dataframe para comparar los resultados
medidas= pd.DataFrame(index=['mse','rmse','mae','mape','max'])

#Configuracion de Tecnicas
**Árbol de Regresión**
(DecisionTreeRegressor)

criterion='squared_error', min_samples_leaf=2, max_depth=None.

Modelo no lineal, interpreta fácilmente interacciones, pero puede sobreajustar si no se limita la profundidad.

**Random Forest** (RandomForestRegressor)

n_estimators=100, max_samples=0.9, criterion='squared_error', max_depth=None, min_samples_leaf=2.

Ensamble de árboles con bagging; reduce varianza del árbol individual y capta relaciones no lineales.

**KNN** (KNeighborsRegressor)

n_neighbors=1, metric='euclidean'.

Predice por vecindad. Requiere datos en la misma escala; con k=1 suele tener alta varianza (muy sensible al ruido).

**Red Neuronal** (MLPRegressor)

activation='relu', hidden_layer_sizes=(16,), learning_rate='constant', learning_rate_init=0.3, momentum=0.2, max_iter=500, random_state=3.

Modelo no lineal flexible; el loss curve mostró la convergencia del entrenamiento.

**ARBOL**

In [0]:
#Creación del modelo con el conjunto de entrenamiento
from sklearn.tree import DecisionTreeRegressor
model_Tree = DecisionTreeRegressor(criterion='squared_error', min_samples_leaf=2, max_depth=None)
model_Tree.fit(X_train, Y_train)#70% entrenamiento

In [0]:
#Graficar el árbol
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
nombres_variables=X_train.columns.values
plt.figure(figsize=(20,20))
plot_tree(model_Tree, feature_names=nombres_variables, filled=True,fontsize=8)
plt.show()

In [0]:
#Evaluación del árbol 30%
from sklearn import metrics
import numpy as np
Y_pred = model_Tree.predict(X_test) #30%

#Medidas de evaluación en regresión
mse = metrics.mean_squared_error(Y_test,Y_pred)
rmse = np.sqrt(mse)
mae= metrics.mean_absolute_error(Y_test,Y_pred)
mape=metrics.mean_absolute_percentage_error(Y_test,Y_pred)
max=metrics.max_error(Y_test,Y_pred)
medidas['Arbol']=[mse, rmse, mae, mape,max]
medidas

In [0]:
#Gráfica Valor Real vs Predicción
plt.scatter(Y_test, Y_pred)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()],'k--', color = 'black', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Valor del modelo')
plt.title('Valor Real vs Predicción Tree')
plt.show() # Mostrar la grafica luego de que ya se definio todos los elementos

**RANDOM FOREST**

In [0]:
#Random Forest
from sklearn.ensemble import RandomForestRegressor

model_rf= RandomForestRegressor(n_estimators=100,  max_samples=0.9, criterion='squared_error',
                              max_depth=None, min_samples_leaf=2)
model_rf.fit(X_train, Y_train) #70%

In [0]:
#Evaluación de Random
from sklearn import metrics

Y_pred = model_rf.predict(X_test) #30%

#Medidas de error
mse = metrics.mean_squared_error(Y_test,Y_pred)
rmse = np.sqrt(mse)
mae= metrics.mean_absolute_error(Y_test,Y_pred)
mape=metrics.mean_absolute_percentage_error(Y_test,Y_pred)
max=metrics.max_error(Y_test,Y_pred)
medidas['RF']=[mse, rmse, mae, mape,max]
print(medidas)

#Gráfica Valor Real vs Predicción
plt.scatter(Y_test, Y_pred)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()],'k--', color = 'black', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Valor del modelo')
plt.title('Valor Real vs Predicción Random')
plt.show()

**KNN**

In [0]:
from sklearn.neighbors import  KNeighborsRegressor
model_Knn = KNeighborsRegressor(n_neighbors=1, metric='euclidean') #minkowski
model_Knn.fit(X_train, Y_train) #70%

In [0]:
#Evaluación de KNN
from sklearn import metrics

Y_pred = model_Knn.predict(X_test) #30%

#Medidas de error
mse = metrics.mean_squared_error(Y_test,Y_pred)
rmse = np.sqrt(mse)
mae= metrics.mean_absolute_error(Y_test,Y_pred)
mape=metrics.mean_absolute_percentage_error(Y_test,Y_pred)
max=metrics.max_error(Y_test,Y_pred)
medidas['Knn']=[mse, rmse, mae, mape,max]
print(medidas)

#Gráfica Valor Real vs Predicción
plt.scatter(Y_test, Y_pred)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()],'k--', color = 'black', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Valor del modelo')
plt.title('Valor Real vs Predicción Knn')
plt.show()

**RED NEURONAL**

In [0]:
from sklearn.neural_network import MLPRegressor

#Solo se configura capas ocultas, no se configura capa de entrada y de salida
#activation -> función activación de la oculta: sigmoid, logistic, linear, relu
#hidden_layer_sizes=5,7 -> dos capas ocultas con 5 neuronas y 7 neuronas
#learning_rate-> tamaño del paso constante o decreciente (constant, adaptive)
#learning_rate_init-> valor tasa de aprendizaje
#momentum-> valor momentum
#max_iter-> iteaciones
#random_state-> semilla para generacion numeros seudoaletorios

model_NN = MLPRegressor(activation="relu",hidden_layer_sizes=(16), learning_rate='constant',
                     learning_rate_init=0.3, momentum= 0.2, max_iter=500,  random_state=3)

model_NN.fit(X_train, Y_train)#70%

In [0]:
#Loss es la desviación entre Y_train y el Y_pred
loss_values = model_NN.loss_curve_
plt.plot(loss_values)

In [0]:
#Evaluación de NN
from sklearn import metrics

Y_pred = model_NN.predict(X_test) #30%

#Medidas de error
mse = metrics.mean_squared_error(Y_test,Y_pred)
rmse = np.sqrt(mse)
mae= metrics.mean_absolute_error(Y_test,Y_pred)
mape=metrics.mean_absolute_percentage_error(Y_test,Y_pred)
max=metrics.max_error(Y_test,Y_pred)
medidas['NN']=[format(mse), rmse, mae, mape,max]
print(medidas)

#Gráfica Valor Real vs Predicción
plt.scatter(Y_test, Y_pred)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()],'k--', color = 'black', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Valor del modelo')
plt.title('Valor Real vs Predicción Red Neuronal')
plt.show()

#Hiperparametrizacion
Se decide finalmente Random Forest como modelo final por precisión y robustez 

In [0]:
# Random Forest - GridSearch simple (regresión)
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Modelo base
rf = RandomForestRegressor(
    criterion='squared_error',
    max_samples=0.9,
    random_state=42,
    n_jobs=-1
)

# Hiperparámetros a buscar
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5],
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",  # RMSE (negativo por convención de sklearn)
    cv=3,
    n_jobs=-1,
    verbose=0
)

# Ajuste
grid.fit(X_train, np.ravel(Y_train))

# Mejor modelo
best_rf = grid.best_estimator_
print("Mejores parámetros:", grid.best_params_)
print("Mejor RMSE (CV):", -grid.best_score_)

In [0]:
#Random Forest
from sklearn.ensemble import RandomForestRegressor

model_rf= RandomForestRegressor(n_estimators=500,  max_samples=0.9, criterion='squared_error',
                              max_depth=10, min_samples_leaf=2)
model_rf.fit(X_train, Y_train) #70%

In [0]:
# Crear un dataframe de datos futuros (con las columnas categóricas originales)
future_data = pd.DataFrame({
    'votes': [1000, 2500, 5000],
    'gross': [1e6, 2.5e6, 5e6],
    'runtime': [90, 120, 150],
    'rating': ['PG', 'R', 'PG-13'],
    'genre': ['Comedy', 'Action', 'Drama'],
    'director': ['Ridley Scott', 'Woody Allen', 'Steven Spielberg'],
    'star': ['Tom Cruise', 'Robert De Niro', 'Tom Hanks'],
    'company': ['Warner Bros.', 'Universal Pictures', 'Paramount Pictures']
})

# Crear dummies igual que en el entrenamiento
future_data = pd.get_dummies(future_data, columns=['rating', 'genre', 'director', 'star', 'company'], drop_first=False, dtype=int)

# Reindexar para asegurar que tenga las mismas columnas que X_train (faltantes se llenan con 0)
future_data = future_data.reindex(columns=X_train.columns, fill_value=0)

# Normalizar variables numéricas igual que en el entrenamiento
future_data[variables_a_normalizar] = min_max_scaler.transform(future_data[variables_a_normalizar])

# Predecir con el modelo Random Forest entrenado
future_pred = model_rf.predict(future_data)

# Mostrar resultados
future_results = future_data.copy()
future_results['score_pred'] = future_pred
print(future_results)

In [0]:
#Grafica de las scores de cada pelicula 
plt.bar(future_results['score_pred'], future_results['score_pred'])
plt.xlabel('Película')
plt.ylabel('Score')
plt.title('Score de cada película')
plt.show()

In [0]:
displayHTML(""""<font size="6" color="blue" face="sans-serif">Dashboard de MOVIES IMDB</font> """)

In [0]:
displayHTML(""""<font size="5" color="blue" face="sans-serif">Mineria Descriptiva</font> """)

In [0]:
displayHTML(""""<font size="5" color="blue" face="sans-serif">Mineria Predictiva</font> """)